<a href="https://colab.research.google.com/github/Santosdevbjj/FraudGraph-Brasil-fraudes-digitais/blob/main/notebooks/exploracao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 FraudGraph Brasil - Análise Exploratória de Grafos (EDA)

Este notebook serve como o ambiente de experimentação e validação de hipóteses do projeto **FraudGraph Brasil**, seguindo as diretrizes de desenvolvimento profissional. Aqui testamos a conectividade com o **Neo4j Aura Cloud**, exploramos a densidade dos relacionamentos e validamos as queries Cypher que sustentam o nosso Agente de IA.

In [13]:

# 1. Instalação de Dependências no Servidor do Colab
!pip install neo4j langchain langchain-community langchain-openai python-dotenv

import os
import sys

# 2. Gerenciamento Inteligente de Chaves de Acesso (Colab Secrets vs Local .env)
try:
    from google.colab import userdata
    os.environ["NEO4J_URI"] = userdata.get('NEO4J_URI')
    os.environ["NEO4J_USER"] = "neo4j"
    os.environ["NEO4J_PASSWORD"] = userdata.get('NEO4J_PASSWORD')
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    print("✅ Chaves de acesso carregadas com sucesso via Colab Secrets!")
except Exception:
    # Fallback automático caso o notebook seja baixado e rodado localmente no VS Code
    from dotenv import load_dotenv
    load_dotenv(dotenv_path='../.env')
    print("ℹ️ Painel do Colab não detectado. Carregando chaves via arquivo .env...")

# 3. Configuração do Sistema de Caminhos para Importação do Projeto
# Garante que o interpretador do Python localize a pasta raiz do seu repositório (src)
sys.path.append(os.path.abspath(os.path.join('..')))

# 4. Importação dos Módulos Customizados do FraudGraph
try:
    from src.database.neo4j_connection import Neo4jConnection
    print("✅ Módulos estruturais do FraudGraph importados com sucesso!")
except ModuleNotFoundError:
    print("⚠️ Módulos locais 'src' não encontrados no diretório atual do Colab.")
    print("Certifique-se de que o ambiente de execução está correto ou insira as classes manualmente se necessário.")

ℹ️ Painel do Colab não detectado. Carregando chaves via arquivo .env...
⚠️ Módulos locais 'src' não encontrados no diretório atual do Colab.
Certifique-se de que o ambiente de execução está correto ou insira as classes manualmente se necessário.


---
## 🔌 1. Testando Conexão com o Banco de Dados

In [14]:

try:
    db = Neo4jConnection()
    # Teste simples de leitura
    res = db.query("MATCH (n) RETURN count(n) as total_nos")
    print(f"✅ Conexão estabelecida com sucesso! Total de nós no banco: {res[0]['total_nos']}")
except Exception as e:
    print(f"❌ Erro de conexão: {e}")

❌ Erro de conexão: name 'Neo4jConnection' is not defined


---
## 📊 2. Análise Descritiva do Grafo (Volumetria)
Antes de rodar algoritmos complexos, precisamos entender a distribuição de rótulos (*Labels*) e tipos de conexões do nosso ambiente de dados.

In [15]:

# Contagem de nós por tipo de Label

# Garante que a conexão exista, mesmo se você não tiver rodado as células anteriores
if 'db' not in locals() and 'db' not in globals():
    print("🔄 Conexão 'db' não encontrada na memória. Inicializando nova conexão...")
    try:
        db = Neo4jConnection()
        print("✅ Conexão 'db' restabelecida com sucesso!")
    except NameError:
        print("❌ Erro: A classe 'Neo4jConnection' não foi importada na Célula 2.")
        print("Por favor, execute a Célula 2 novamente antes de rodar este bloco.")

# Executa a query caso a conexão esteja ativa
if 'db' in locals() or 'db' in globals():
    label_query = """
    MATCH (n)
    RETURN labels(n)[0] AS TipoNo, count(n) AS Quantidade
    """

    try:
        resultados = db.query(label_query)
        print("📊 Distribuição de Nós no Grafo:")
        print(resultados)
    except Exception as e:
        print(f"❌ Erro ao executar a consulta no Neo4j: {e}")

🔄 Conexão 'db' não encontrada na memória. Inicializando nova conexão...
❌ Erro: A classe 'Neo4jConnection' não foi importada na Célula 2.
Por favor, execute a Célula 2 novamente antes de rodar este bloco.


In [16]:

# Contagem de relacionamentos por tipo de transação

# Garante que a conexão exista, mesmo se o estado do Colab tiver oscilado
if 'db' not in locals() and 'db' not in globals():
    print("🔄 Conexão 'db' não encontrada na memória. Inicializando nova conexão...")
    try:
        from src.database.neo4j_connection import Neo4jConnection
        db = Neo4jConnection()
        print("✅ Conexão 'db' restabelecida com sucesso!")
    except Exception as e:
        print(f"❌ Erro ao tentar restabelecer a conexão: {e}")
        print("Por favor, certifique-se de que a Célula 2 foi executada com sucesso.")

# Executa a query caso a conexão esteja ativa
if 'db' in locals() or 'db' in globals():
    rel_query = """
    MATCH ()-[r]->()
    RETURN type(r) AS TipoRelacionamento, count(r) AS Total
    """

    try:
        resultados_rel = db.query(rel_query)
        print("📊 Volumetria de Relacionamentos no Grafo:")
        print(resultados_rel)
    except Exception as e:
        print(f"❌ Erro ao executar a consulta de relacionamentos no Neo4j: {e}")

🔄 Conexão 'db' não encontrada na memória. Inicializando nova conexão...
❌ Erro ao tentar restabelecer a conexão: No module named 'src'
Por favor, certifique-se de que a Célula 2 foi executada com sucesso.


---
## 🔍 3. Validação de Hipóteses de Fraude (A Query do Hackathon)

### **Hipótese:** Centrais de fraude compartilham o mesmo hardware físico (`Dispositivo`) para movimentar contas de CPFs clonados ou laranjas em direção a uma mesma conta receptora rápida.

In [17]:

# Query de Triangulação de Risco Máximo

# Garante que a conexão exista, impedindo o erro de NameError
if 'db' not in locals() and 'db' not in globals():
    print("🔄 Conexão 'db' não encontrada na memória. Inicializando nova conexão...")
    try:
        from src.database.neo4j_connection import Neo4jConnection
        db = Neo4jConnection()
        print("✅ Conexão 'db' restabelecida com sucesso!")
    except Exception as e:
        print(f"❌ Erro ao tentar restabelecer a conexão: {e}")
        print("Por favor, certifique-se de que a Célula 2 foi executada com sucesso.")

# Executa a query coração do Hackathon caso a conexão esteja ativa
if 'db' in locals() or 'db' in globals():
    fraud_query = """
    MATCH (c:Cliente)-[:UTILIZA]->(d:Dispositivo),
          (c)-[:TRANSFERIU]->(p:ContaDestino)
    WITH d, p, collect(c.nome) as clientes, count(c) as total_cpfs
    WHERE total_cpfs >= 3
    RETURN d.device_id as dispositivo,
           d.ip as ip_dispositivo,
           p.pix as pix_destino,
           clientes,
           total_cpfs
    """

    try:
        print("🔍 Executando varredura de triangulação no Neo4j...")
        resultados_fraude = db.query(fraud_query)

        # Formatação limpa em JSON para legibilidade dos avaliadores
        import json
        print("\n🚨 Padrões Suspeitos Identificados:")
        print(json.dumps(resultados_fraude, indent=2, ensure_ascii=False))

    except Exception as e:
        print(f"❌ Erro ao executar a query de fraude: {e}")

🔄 Conexão 'db' não encontrada na memória. Inicializando nova conexão...
❌ Erro ao tentar restabelecer a conexão: No module named 'src'
Por favor, certifique-se de que a Célula 2 foi executada com sucesso.


---
## 🤖 4. Simulação da Camada do Agente de IA
Vamos simular como o resultado bruto estruturado em formato tabular/JSON do Neo4j é passado para o modelo de linguagem (LLM) interpretar e gerar o parecer de negócio (*Reasoning*).

In [18]:

# 🤖 4. Simulação da Camada do Agente de IA com Raciocínio (Reasoning)

import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# --- Definição Direta dos Módulos (Garante Execução Independente no Colab) ---

SYSTEM_PROMPT = """
Você é um Analista de IA Especialista em Prevenção a Fraudes Bancárias e Segurança Digital.
Sua missão é analisar o resultado de consultas estruturadas de bancos de dados em grafos (Neo4j) e gerar um relatório analítico para a equipe de auditoria/compliance.

Ao analisar os dados fornecidos:
1. Identifique o padrão técnico suspeito (ex: múltiplos CPFs compartilhando o mesmo hardware ou rede).
2. Explique o risco operacional e financeiro desse comportamento baseado em fraudes do mundo real (ex: ataques de Engenharia Social, Dispositivos Clonados, Contas Laranjas).
3. Seja direto, profissional e focado em ações práticas (Business Insights).

Traduza os dados de forma que um Diretor de Operações sem conhecimento técnico em grafos consiga tomar uma decisão rápida de bloqueio ou investigação.
"""

class FraudGraphAgentColab:
    def __init__(self):
        # Recupera a chave de API configurada no ambiente ou no Secrets do Colab
        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key:
            raise ValueError("❌ Erro: A variável 'OPENAI_API_KEY' não foi encontrada no ambiente. Certifique-se de ativar o acesso a ela no painel Secrets (ícone de chave 🔑).")

        self.llm = ChatOpenAI(
            model="gpt-4o",
            temperature=0.2,
            api_key=api_key
        )

    def analyze_fraud_pattern(self, raw_graph_data):
        if not raw_graph_data:
            return "Nenhum padrão anômalo detectado no grafo nas últimas varreduras."

        messages = [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=f"Dados brutos retornados pelo grafo Neo4j: {str(raw_graph_data)}")
        ]

        response = self.llm.invoke(messages)
        return response.content

# --- Execução do Fluxo de Inteligência ---

# Verifica se a célula anterior gerou os dados estruturados de fraude
if 'resultados_fraude' not in locals() and 'resultados_fraude' not in globals():
    print("⚠️ Os dados 'resultados_fraude' não foram encontrados na memória.")
    print("Simulando uma massa de dados de triangulação para o teste da IA...")
    # Massa de dados mockada idêntica à do banco para evitar travar a demonstração
    resultados_fraude = [{
        'dispositivo': 'IPHONE999',
        'ip_dispositivo': '192.168.1.10',
        'pix_destino': 'fraude@pix.com',
        'clientes': ['João Silva', 'Maria Souza', 'Pedro Santos'],
        'total_cpfs': 3
    }]

try:
    # Instancia o agente adaptado para o Colab
    agent = FraudGraphAgentColab()

    print("🤖 Enviando subgrafo anômalo para o Agente de IA...")
    print("🧠 Gerando parecer analítico com capacidade de raciocínio (Reasoning)...\n")

    parecer = agent.analyze_fraud_pattern(resultados_fraude)

    print("="*60)
    print("📋 PARECER TÉCNICO DE SEGURANÇA RECOMENDADO PELA IA:")
    print("="*60)
    print(parecer)
    print("="*60)

except Exception as e:
    print(f"\n❌ Erro durante a execução do Agente de IA: {e}")


❌ Erro durante a execução do Agente de IA: ❌ Erro: A variável 'OPENAI_API_KEY' não foi encontrada no ambiente. Certifique-se de ativar o acesso a ela no painel Secrets (ícone de chave 🔑).
